# Phase 1 experiments: TensorRT-LLM 1.4.0rc3 PyTorch v4

Optional inference parameters and measured-window server telemetry, following the SGLang v4 pattern.
Uses versioned `trtllm_policy_v4.py`, `runtime_trtllm_v4.py`, and `telemetry_trtllm_v4.py`.
Existing notebooks and shared registries are unchanged. Run setup in order in a fresh kernel.
Phase A uses the pinned BF16 source; each Phase B campaign selects one actual BF16 or ModelOpt-exported FP8 checkpoint.
KV-cache precision is independent. Export/calibration stays outside benchmark timing.
CPU simulation checks wiring only; remote CLI/schema and GPU validation are required.


V4 adds frozen datasets and quality to the complete v3 engine workflow. See [the setup guide](datasets_v4.md).

## Engine policy inherited from v3
See [the parity checklist](phase1_v2_parity.md). Independent optional cells below restore Pareto/constraint searches, additional pruners, calibration diagnostics, and pinned HF/speculative setup. The supervised default scalar queue remains unchanged. Untuned baselines and full v2 reporting remain separate. Phase A defaults to the same target as B; explicit HF setup changes only Phase B.


In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "llm_optimizer").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from llm_optimizer.experiment.phase1 import plot_group
from llm_optimizer.experiment.trtllm_policy_v3 import (
    TRTLLMRunnerV3, TRTLLMLabV3, DEFAULT_OPTIONAL_PARAMS,
    native_command, runtime_help, audit_command, resolve_cli, verify_runtime,
)
import copy

import copy
from llm_optimizer.experiment.v4.campaign import (
    CampaignWorkloads, attach_campaign, evaluate_held_out, prepare_legacy,
)
from llm_optimizer.experiment.v4.runtime import Generation
from llm_optimizer.experiment.v4.scorers import ScorerSettings


## VM, model, workload, and budget controls
This notebook launches trtllm on a dedicated VM. Use a separate campaign name for independent replications. Changing any search setting creates a new configuration hash; restarts with identical settings resume existing studies. All campaigns on a VM must share `RUN_ROOT` and the same GPU assignment.
`SAMPLER_NAMES` selects GP (`gp`), TPE (`tpe`), and multivariate TPE (`tpe_multivariate`). GP requires the engine-compatible PyTorch installation; unavailable GP is explicitly reported and recorded as skipped, never replaced by TPE. Each combination of sampler, seed, framework, concurrency, objective and pruner has **20 attempted trials** by default. Failed/pruned attempts count toward that budget. Three samplers and three seeds therefore mean 180 attempts for a single concurrency/objective/framework/pruner, not 20 total. Optional `sampler_names=["gp"]` on a group call restricts that cell.

| Concurrency | Requests per trial | Default trial attempts per study |
|---|---|---|
| 1 | 32 | 20 |
| 8 | 62 | 20 |
| 16 | 125 | 20 |
| 32 | 250 | 20 |
| 128 | 1000 | 20 |

The loaded request/concurrency ratio is approximately 1000/128. Concurrency 1 deliberately uses 32 requests. `REQUESTS_BY_CONCURRENCY` is used by both A and B; repeat validation uses its independent request count (1000 by default). Pruning observation floors scale from v2's 400/1000 ratio: 12, 24, 50, 100, 400 respectively. With small request counts, tail-latency estimates are noisy; the separate validation budget remains necessary.


In [ ]:
CAMPAIGN = "phase1-trtllm-datasets-v4"
RUN_NAME = "trt140rc3-v3"  # W&B display-name prefix; keep campaign unchanged
SAVE_REQUEST_OUTPUTS = True  # persist generated_texts in local *-measured.jsonl files
FRAMEWORKS = ["trtllm"]  # or ["vllm", "sglang", "trtllm"]
RUN_REAL_GPU = False
CONCURRENCIES = [1, 8, 16, 32, 128]
SEEDS = [11, 29, 47]
RUN_ROOT = ROOT / "experiment_results" / "phase1-datasets-v4"
CAMPAIGN_HOURS = 168
GROUP_TIMEOUT_SECONDS = 6 * 3600  # includes all frameworks/seeds in a group
MAX_RESTARTS = 2  # bounded retries after a crashed/timed-out group
MODEL = "Qwen/Qwen3.5-4B"  # same model across A/B isolates search-space effects
MODEL_SNAPSHOT = "/workspace/models/pinned-model-snapshot"
TRIALS = 20  # attempts PER sampler/seed/framework/concurrency/pruner
SAMPLER_NAMES = ["tpe", "tpe_multivariate"]  # select any nonempty subset
PARETO_SAMPLERS = [*SAMPLER_NAMES, "nsga2"]  # available to optional Pareto calls
REQUESTS_BY_CONCURRENCY = {1: 32, 8: 62, 16: 125, 32: 250, 128: 1000}
# Preserve the v2 400/1000 pruning-observation fraction at each request budget.
PRUNING_MIN_COMPLETED_REQUESTS = {
    c: max(1, int(n * 0.4)) for c, n in REQUESTS_BY_CONCURRENCY.items()
}
SAMPLER_SETTINGS = {
    "startup_single": 4, "startup_multi": 10,
    "n_ei_candidates": 24, "population_size": 8,
}
VALIDATION_REPEATS = 5

# W&B controls match optuna_inference_optimization_lab_v2.
WANDB_ENABLED = False
WANDB_PROJECT = "inference-optimization"
WANDB_ENTITY = "maorblumberg-tel-aviv-university"  # or None for your SDK default
WANDB_MODE = "online"  # "offline" keeps runs local; False above disables the SDK
WANDB_GROUP = CAMPAIGN  # same label across framework VMs

# Fill with actual verified values. Never mark checks true without performing them.
PREFLIGHT = {
    framework: {
        "gpu_type": "REPLACE", "framework_version": "REPLACE",
        "cuda_version": "REPLACE", "driver_version": "REPLACE",
        "model_revision": "REPLACE_WITH_MODEL_COMMIT",
        "native_config_verified": False, "quality_verified": False,
        "verified_precision_combinations": [],  # weights/cache pairs approved on this VM
    }
    for framework in ["vllm", "sglang", "trtllm"]
}


# Workload and campaign controls
RUN_MODEL_SETUP = False  # resolve/download the pinned model explicitly
RUN_NATIVE_AUDIT = False
RUN_STUDIES = False
RUN_VALIDATION = False  # v3 candidate repeats
RUN_HELD_OUT = False
HELD_OUT_STUDY_NAME = None
HELD_OUT_PARETO_TRIAL = None
DATA_SOURCE = "fixture"  # fixture, hf, local, or legacy_random
SOURCE_FORMAT = "extraction"  # sharegpt, messages, alpaca, gpqa_diamond, humaneval, humaneval_plus
DATASET_ID = None  # Hub ID or path to JSON/JSONL
DATASET_REVISION = None  # immutable Hub commit for DATA_SOURCE="hf"
DATASET_SPLIT = "train"
DATASET_CONFIG = None
MODEL_REVISION = "REPLACE_WITH_MODEL_COMMIT"
GENERATION = Generation(output_tokens=256, eos="stop", thinking="default")
THINKING_TEMPLATE_VERIFIED = False
SCORER = ScorerSettings()
SCORE_QUALITY = True
QUALITY_THRESHOLDS = ()  # e.g. (("accuracy", 0.60),)
SAMPLING_SEED = 2026
TOKEN_BUCKETS_BY_CONCURRENCY = {}  # {8: ((0, 512, 20), (512, 1024, 42))}
HELD_OUT_COUNT = 32
DATASET_REPEAT_REQUESTS = 1000
VALIDATION_SEED = 50000
LEGACY_EOS = "ignore"  # original v3 client default
LEGACY_INPUT_TOKENS, LEGACY_OUTPUT_TOKENS = 512, 128
LEGACY_DATASET_PATH = None  # optional local copy of the original ShareGPT JSON
LEGACY_DATASET_REVISION = "192ab2185289094fc556ec8ce5ce1e8e587154ca"
DATASETS = None  # populated by preparation after model setup


## W&B login and study logging
Install the SDK with `pip install -e '.[notebook,wandb]'`. Run the login cell before creating studies, as in `optuna_inference_optimization_lab_v2.ipynb`. Use `WANDB_ENABLED=False` to disable tracking, or `WANDB_MODE="offline"` to save runs locally without login; upload offline runs later with `wandb sync`.

The existing shared study logger records sampled parameters (including precision), rendered engine configs, workload/environment contracts, benchmark/objective metrics, request accounting, failure/pruning details, cumulative GPU cost, and final best/Pareto candidates. One W&B run is created per study invocation, named after its stable Optuna study name and grouped by `WANDB_GROUP`. Synthetic runs carry the `SYNTHETIC` tag. W&B logging failures warn and allow optimization to continue. Independent validation remains in local artifacts, matching v2.

For unattended runs, authenticate on the VM first with `wandb login` or provide `WANDB_API_KEY` through the service environment. Keep credentials outside notebook cells and experiment configs. The batch supervisor authenticates before launching the queue; CPU smoke tests disable tracking.


In [ ]:
if WANDB_ENABLED and WANDB_MODE == "online":
    import wandb
    wandb.login()  # uses existing credentials/WANDB_API_KEY or prompts interactively


## Explicit base and extended search spaces
Every domain below is editable. Base numeric domains must be regularly spaced and contain at least two values. Extended categorical domains must be supported by the installed engine and model. Remove unsupported choices after preflight, including numerical quality checks for cache dtype changes. Graph profiles and padding are conditional. `schedule_conservativeness` is SGLang-only; Phase A retains its fixed baseline value of 1.0, while Phase B searches the explicit domain below.

In [ ]:
# Illustrative SLOs; replace after measuring the pinned model/workload/GPU.
TTFT_LIMITS_MS = {"1": 100.0, "128": 200.0}
E2E_LIMITS_MS = {"1": 2000.0, "128": 6000.0}
FAILURE_RATE_LIMIT = 0.0
CONSTRAINED = False
PRUNER_NAMES = ["nop", "median", "percentile", "halving", "hyperband", "threshold"]
PRUNING_SAMPLES = 32
# Ordinary serving is explicit. Setup is opt-in and may download checkpoints.
HF_SETUP = {
    "target_model_id": MODEL, "target_revision": MODEL_REVISION,
    "framework": "trtllm", "enable_dflash": False,
    "speculative_algorithm": None, "draft_model_id": None, "draft_revision": None,
    "block_size": 16, "num_speculative_tokens": 15, "max_draft_len": 3,
}
# SGLang: DFLASH/DSPARK; vLLM: enable_dflash=True selects DSpark.
# TRT: explicit algorithms accepted by predefined.trtllm.validate_draft_request.
# Configure a separate campaign/run root to compare ordinary and speculative serving.
MODEL_PROFILES = {}  # populated only by explicit setup_pinned_models()
MODEL_SETUP_IDENTITY = None
MODEL_SETUP_CHOICES = None
PHASE_B_MODEL = None
PHASE_B_SNAPSHOT = None


In [ ]:
PREFILL_BUDGETS = list(range(1024, 8193, 1024))
TOKEN_BUDGETS = list(range(1024, 32769, 256))
CAPACITIES = list(range(8, 129, 8))
BASE_SPACES = {
    1: {
        "vllm": {"max_num_batched_tokens": PREFILL_BUDGETS, "cuda_graph_on": [0, 1]},
        "sglang": {"chunked_prefill_size": PREFILL_BUDGETS, "cuda_graph_on": [0, 1]},
        "trtllm": {"max_num_tokens": PREFILL_BUDGETS, "cuda_graph_on": [0, 1]},
    },
    128: {  # shared regime for all concurrency levels > 1
        "vllm": {
            "max_num_seqs": CAPACITIES, "max_num_batched_tokens": TOKEN_BUDGETS,
            "gpu_memory_utilization": [0.75, 0.80, 0.85, 0.90],
        },
        "sglang": {
            "max_running_requests": CAPACITIES, "chunked_prefill_size": TOKEN_BUDGETS,
            "mem_fraction_static": [0.70, 0.75, 0.80, 0.85],
        },
        "trtllm": {
            "max_batch_size": CAPACITIES, "max_num_tokens": TOKEN_BUDGETS,
            "kv_cache_free_gpu_memory_fraction": [0.65, 0.75, 0.85, 0.95],
        },
    },
}

EXPANDED_CHOICES = {'vllm': {'async_scheduling': [False, True],
          'performance_mode': ['balanced', 'interactivity', 'throughput'],
          'attention_backend': ['auto', 'FLASH_ATTN', 'FLASHINFER'],
          'kv_cache_dtype': ['auto'],
          'enable_prefix_caching': [False, True],
          'graph_mode': ['NONE', 'PIECEWISE', 'FULL_AND_PIECEWISE']},
 'sglang': {'attention_backend': ['triton', 'fa3', 'flashinfer'],
            'disable_radix_cache': [False, True],
            'schedule_policy': ['fcfs', 'lpm'],
            'enable_mixed_chunk': [False, True],
            'disable_overlap_schedule': [False, True],
            'cuda_graph_backend_decode': ['disabled', 'full'],
            'cuda_graph_backend_prefill': ['disabled', 'breakable']},
 'trtllm': {'attn_backend': ['TRTLLM'],
            'enable_chunked_prefill': [False, True],
            'enable_block_reuse': [False, True],
            'capacity_scheduler_policy': ['GUARANTEED_NO_EVICT', 'MAX_UTILIZATION'],
            'graph_enabled': [False, True]}}

EXTRA_SPACES = {
    "vllm": {"graph_capture_profile": ["powers", "dense"]},
    "sglang": {
        "schedule_conservativeness": [0.5, 0.75, 1.0, 1.25, 1.5],
        "max_prefill_tokens": [8192, 16384, 24576, 32768],
    },
    "trtllm": {
        "graph_capture_profile": ["powers", "dense"],
        "graph_padding": [0, 1],
    },
}


## Optional parameters and measured-window telemetry
All optional parameters start disabled. Enable a small remotely verified subset
before creating studies; singleton values provide fixed Phase B overrides.
Optional settings apply only to Phase B. Changing options, telemetry or versioned
implementation files creates a new campaign hash.

- Early first token requires overlap; in-graph sampling requires decode graphs.
- Prefill captures and compilation merit separate campaigns because startup and
  memory costs can dominate 20 attempts. Breakable conflicts with torch.compile,
  LoRA and multimodal execution; BF16 Qwen3.5 coverage does not establish FP8 support.
- `prefill_capture_profile`: workload captures the prompt length; sparse adds
  512/1024/2048/4096/8192 buckets bounded by the token budget.
- `compile_only_piecewise_graphs` applies only to piecewise prefill graphs.
- KV page sizes require model/backend GPU checks. Streaming interval changes
  delivery cadence. Worker counts consume CPU processes and memory.
- Decode powers now cover capacity, including 64/128; concurrency one skips
  ineffective graph-profile and padding dimensions.

The collector samples `trtllm_num_requests_running`, `trtllm_num_requests_waiting`
and `trtllm_kv_cache_utilization` from `/metrics`. Iteration statistics are enabled
and one HTTP frontend is enforced when telemetry is on. This adapter supports
single-GPU telemetry; running/waiting are rank-0-owned totals, not client concurrency.
KV utilization is used/max cache blocks, including retained state, and is not a
complete recurrent-state utilization metric for hybrid models. GPU utilization
uses NVML device busy percent; set an explicit verified physical `gpu_index`.
Unsupported gauges, unknown labels and scrape failures are null with reasons.

Time-weighted averages use left holds over covered time, expiring after two
sampling intervals. Peaks are sampled peaks. Coverage, sample counts and units
are saved. Only the measured-request window is included, excluding startup,
warmup, cleanup and rest. Server publication may be slower than scraping.
Telemetry, client in-flight requests and request throughput persist in JSONL,
execution sidecars, Optuna attributes and W&B; failed/pruned attempts keep partial
execution telemetry. CPU simulation reports unavailable server telemetry.

Official source: [1.4.0rc3 schemas](https://github.com/NVIDIA/TensorRT-LLM/blob/9e334fbaae504cc836cddea8a20cc1bdab543e55/tensorrt_llm/llmapi/llm_args.py),
[metrics](https://github.com/NVIDIA/TensorRT-LLM/blob/9e334fbaae504cc836cddea8a20cc1bdab543e55/tensorrt_llm/metrics/collector.py).


In [ ]:
# Enable only the additional remotely verified controls needed for this campaign.
optional_params = copy.deepcopy(DEFAULT_OPTIONAL_PARAMS)
# Example: optional_params["enable_early_first_token_response"]["enabled"] = True
TELEMETRY = {
    "enabled": True,
    "interval_s": 0.5,
    "timeout_s": 0.25,
    "gpu_index": None,  # Explicit physical NVML index; never inferred from CUDA visibility.
}
optional_params


## Bind the shared runner
Persistent SQLite storage is local to each VM. Copy artifacts off the VM periodically. A VM-wide file lock prevents overlapping workers under the same root. The runner records abandoned RUNNING trials as FAIL after acquiring exclusive ownership; failed and pruned attempts count toward the configured trial budget. Notebook kernels do not have the batch script’s process-group watchdog. After an interrupted notebook, ensure its server processes have stopped before resuming.
Every group invocation reads the current controls and resumes via SQLite. Editing the request map or sampler selection produces a separate campaign configuration hash.


## Weight and KV-cache precision
Phase A uses BF16 source weights. Phase B defaults to FP8 weights + FP8 KV cache,
loading a separate unified Hugging Face checkpoint exported from that exact source.
Calibration/export runs once outside timed trials. No quant_config/calib_config startup
arguments are sent to TRT-LLM 1.4.0rc3.

To compare weights, run Phase B with a singleton weight domain `["bf16"]`, then
`["fp8"]`; each gets a different campaign directory. KV-cache precision is independently
sampled. Explicit `bfloat16` cache overrides FP8 metadata inherited from the export.
Preflight must approve every requested combination after actual GPU/quality checks.


In [ ]:
# Phase B uses one exported weight precision per campaign: ["fp8"] or ["bf16"].
# KV-cache precision remains independent. Run separate campaigns to compare weights.
# Phase A always uses unquantized BF16 weights and the default cache (BF16).
PRECISION_CHOICES = {
    "vllm": {"weight_precision": ["bf16", "fp8"], "kv_cache_precision": ["bf16", "fp8"]},
    "sglang": {"weight_precision": ["bf16", "fp8"], "kv_cache_precision": ["bf16", "fp8"]},
    "trtllm": {"weight_precision": ["fp8"], "kv_cache_precision": ["fp8"]},
}


## JSON export and response capture
Run the next cell before constructing runners. It fixes JSON exports containing `inf`,
`-inf` or `nan` without modifying the stored Optuna trials. To finish the previous studies,
set `SAVE_REQUEST_OUTPUTS=False` and use exactly the previous campaign/model configuration.

With `SAVE_REQUEST_OUTPUTS=True` (the default here), generated text is retained after each
saved benchmark in `*-measured.jsonl`, under `results.generated_texts`. This setting changes
the frozen configuration and creates a fresh results directory under the same campaign.
Changing `RUN_NAME` alone does not reset studies. Text from previous runs cannot be recovered.
Responses are saved locally, not automatically uploaded to W&B, and are available after the
benchmark returns rather than streamed live. Attempts interrupted by errors or pruning may
not produce a result row. The random-token workload is for performance, not quality validation.


In [ ]:
# Notebook-only JSON export fix: preserves SQLite values; sanitizes exported copies.
import json
import math
from pathlib import Path
import llm_optimizer.experiment.phase1 as phase1_module
import llm_optimizer.experiment.workflow as workflow_module

def clean_export_json(value):
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, dict):
        return {key: clean_export_json(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean_export_json(item) for item in value]
    return value

def safe_write_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(clean_export_json(value), indent=2, allow_nan=False) + "\n"
    )
    temporary.replace(path)

# phase1 imported write_json by name, so patch both references in this kernel.
phase1_module.write_json = safe_write_json
workflow_module.write_json = safe_write_json


In [ ]:
# Notebook-only TRT-LLM 1.4.0rc3 PyTorch compatibility adapter.
import copy
from types import MethodType
from importlib.metadata import version
from llm_optimizer.experiment.phase1 import ExplicitLab

TRT140_COMPATIBILITY = {
    "adapter": "notebook-trt140rc3-fp8-checkpoint-v3",
    "framework_version": "1.4.0rc3",
    "backend": "pytorch",
    "weights": "bf16-or-modelopt-fp8-checkpoint",
    "kv_cache_dtypes": ["auto", "fp8"],
}

def notebook_precision_config(self, framework, weights, cache):
    if framework != "trtllm":
        return ExplicitLab.precision_config(self, framework, weights, cache)
    if weights not in {"bf16", "fp8"}:
        raise ValueError("Select bf16 or fp8 weights")
    # Precision is implemented by the phase's checkpoint, not a serving flag.
    # Phase A calls this with bf16. Phase B's singleton weight domain is
    # checked against its checkpoint before the runner is constructed.
    return {"dtype": "bfloat16"}

def validate_trt140_environment():
    verify_runtime()

def resolve_notebook_trt_cli():
    return resolve_cli()

def notebook_serialize_engine_config(self, framework, concurrency, config):
    return TRTLLMLabV3.serialize_engine_config(self, framework, concurrency, config)

# Retain original precision/SLO/fixed-option checks; extend only the quantized
# Phase B checkpoint case owned by this notebook's compatibility adapter.
import llm_optimizer.experiment.phase1 as _phase1
_original_verify_precision = getattr(_phase1.verify_precision, "_before_trt_fp8", _phase1.verify_precision)

def notebook_verify_precision(values, precision_choices):
    checked = copy.deepcopy(values)
    profile = values.get("real_preflight", {}).get("B", {}).get("trtllm", {})
    metadata = profile.get("notebook_compatibility", {})
    if (values.get("frameworks") == ["trtllm"] and
            metadata.get("adapter") == TRT140_COMPATIBILITY["adapter"] and
            values.get("run_real_gpu")):
        domain = precision_choices["trtllm"]["weight_precision"]
        if len(domain) != 1:
            raise ValueError("One weight checkpoint per campaign: choose ['bf16'] or ['fp8']")
        if domain == ["fp8"]:
            artifact = validate_fp8_checkpoint(values["model_snapshots"]["B"],
                                               values["model_snapshots"]["A"])
            if profile.get("weight_artifact") != artifact:
                raise ValueError("FP8 checkpoint identity differs from frozen preflight metadata")
            # The shared check still verifies the unquantized pinned source,
            # requested approved pairs, and all fixed precision conflicts.
            checked["model_snapshots"]["B"] = values["model_snapshots"]["A"]
    return _original_verify_precision(checked, precision_choices)

notebook_verify_precision._before_trt_fp8 = _original_verify_precision
_phase1.verify_precision = notebook_verify_precision


In [ ]:
# FP8 checkpoint preparation controls. Keep source snapshots and exports on persistent storage.
TRT_FP8_CHECKPOINT = ROOT / "models" / "trt-modelopt-fp8"
MODELOPT_ROOT = Path.home() / "Model-Optimizer"  # existing checkout matching installed modelopt
RUN_FP8_EXPORT = False  # set True once to calibrate/export; False reuses a completed export
FP8_CALIB_SAMPLES = 128
FP8_CALIB_SEQUENCE_LENGTH = 512
FP8_CALIB_DATASET = "cnn_dailymail"

import hashlib
import struct

def _file_sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def _fp8_export_metadata(directory):
    directory = Path(directory)
    config = json.loads((directory / "config.json").read_text())
    quant = config.get("quantization_config") or {}
    auxiliary = directory / "hf_quant_config.json"
    if auxiliary.exists():
        quant = {"config": quant, "hf_quant_config": json.loads(auxiliary.read_text())}
    if "FP8" not in json.dumps(quant).upper():
        raise ValueError("Export metadata does not identify FP8 quantization")
    weights = sorted(directory.glob("*.safetensors"))
    if not weights:
        raise ValueError("No exported safetensors weights found")
    headers = []
    has_fp8 = False
    for path in weights:
        with path.open("rb") as handle:
            header_length = struct.unpack("<Q", handle.read(8))[0]
            if header_length > 100_000_000:
                raise ValueError("Invalid safetensors header")
            raw = handle.read(header_length)
            header = json.loads(raw)
        has_fp8 |= any(isinstance(v, dict) and str(v.get("dtype", "")).startswith("F8")
                       for k, v in header.items() if k != "__metadata__")
        headers.append([path.name, path.stat().st_size, hashlib.sha256(raw).hexdigest()])
    if not has_fp8:
        raise ValueError("Export has no FP8-stored tensors; refusing to relabel BF16 weights")
    identity = {"config_sha256": _file_sha(directory / "config.json"),
                "quantization": quant, "weight_headers": headers}
    identity["artifact_id"] = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    return identity

def validate_fp8_checkpoint(directory, source):
    directory, source = Path(directory).resolve(), Path(source).resolve()
    manifest_path = directory / "phase1-fp8-export.json"
    if not manifest_path.is_file():
        raise ValueError("Run the FP8 preparation cell first; completed export manifest is missing")
    manifest = json.loads(manifest_path.read_text())
    if manifest["source_snapshot"] != str(source) or manifest["source_config_sha256"] != _file_sha(source / "config.json"):
        raise ValueError("FP8 export belongs to a different pinned BF16 source")
    identity = _fp8_export_metadata(directory)
    if identity != manifest["checkpoint"]:
        raise ValueError("Export changed since completion; use a new export directory")
    return manifest


In [ ]:
def phase_b_artifact(for_model_setup=False):
    _phase_b_model, _phase_b_snapshot = PHASE_B_MODEL or MODEL, PHASE_B_SNAPSHOT or MODEL_SNAPSHOT
    if not for_model_setup:
        _weights = PRECISION_CHOICES["trtllm"]["weight_precision"]
        if _weights not in (["bf16"], ["fp8"]):
            raise ValueError("Run separate BF16 and FP8 weight campaigns; choose a singleton domain")
        MODEL_PROFILES.get("trtllm", {}).pop("weight_artifact", None)
        if _weights == ["fp8"]:
            _artifact = validate_fp8_checkpoint(TRT_FP8_CHECKPOINT, MODEL_SNAPSHOT)
            _phase_b_snapshot = str(Path(TRT_FP8_CHECKPOINT).resolve())
            _phase_b_model = MODEL + "@modelopt-fp8-" + _artifact["checkpoint"]["artifact_id"][:16]
            MODEL_PROFILES["trtllm"]["weight_artifact"] = _artifact
        else:
            _phase_b_model, _phase_b_snapshot = MODEL, MODEL_SNAPSHOT
    return _phase_b_model, _phase_b_snapshot

def make_runner(for_model_setup=False):
    if MODEL_SETUP_IDENTITY is not None and MODEL_SETUP_IDENTITY != HF_SETUP:
        raise ValueError("HF_SETUP changed; clear MODEL_PROFILES and MODEL_SETUP_IDENTITY, then rerun setup")
    if MODEL_SETUP_CHOICES is not None and MODEL_SETUP_CHOICES != EXPANDED_CHOICES[HF_SETUP["framework"]]:
        raise ValueError("Expanded choices changed after HF setup; clear setup profiles and rerun setup")
    _phase_b_model, _phase_b_snapshot = phase_b_artifact(for_model_setup)
    settings = {
        "ttft_limits_ms": TTFT_LIMITS_MS, "e2e_limits_ms": E2E_LIMITS_MS,
        "failure_rate_limit": FAILURE_RATE_LIMIT, "pruning_samples": PRUNING_SAMPLES,
        "constrained": CONSTRAINED,
        "wandb_enabled": WANDB_ENABLED,
        "wandb_project": WANDB_PROJECT, "wandb_entity": WANDB_ENTITY,
        "wandb_mode": WANDB_MODE, "wandb_group": WANDB_GROUP,
        "frameworks": FRAMEWORKS, "real_frameworks": FRAMEWORKS,
        "run_real_gpu": RUN_REAL_GPU and not for_model_setup, "quick_demo": False,
        "phases": ["A", "B"], "concurrencies": CONCURRENCIES,
        "seeds": SEEDS, "sampler_names": SAMPLER_NAMES, "pareto_samplers": PARETO_SAMPLERS,
        "phase_models": {"A": MODEL, "B": _phase_b_model},
        "model_snapshots": {"A": MODEL_SNAPSHOT, "B": _phase_b_snapshot},
        # Lab's phase-level fallback; actual study counts use the map below.
        "phase_requests": {"A": REQUESTS_BY_CONCURRENCY[max(CONCURRENCIES)],
                           "B": REQUESTS_BY_CONCURRENCY[max(CONCURRENCIES)]},
        "sampler_settings": SAMPLER_SETTINGS,
        "search_trials": {p: {"1": TRIALS, "128": TRIALS} for p in ["A", "B"]},
        "base_workload": {
            **({"output_details": True} if SAVE_REQUEST_OUTPUTS else {}),
            "dataset_name": "random", "apply_chat_template": True, "random_input_len": LEGACY_INPUT_TOKENS,
            "random_output_len": LEGACY_OUTPUT_TOKENS, "random_range_ratio": 1.0,
            "seed": 2026, "request_rate": "inf",
            **({"disable_ignore_eos": True}
               if DATA_SOURCE == "legacy_random" and LEGACY_EOS == "stop" else {}),
        },
        "validation_repeats": VALIDATION_REPEATS,
        "validation_seed": VALIDATION_SEED,
        "validation_requests": dict.fromkeys(
            ("A", "B"), 1000 if DATA_SOURCE == "legacy_random" else DATASET_REPEAT_REQUESTS
        ),
        "gpu_count": 1, "rest_seconds": 10,
        "real_preflight": {
            phase: {f: {**PREFLIGHT[f], "expanded_choices": EXPANDED_CHOICES[f],
                           **(MODEL_PROFILES.get(f, {}) if phase == "B" else {})}
                    for f in FRAMEWORKS}
            for phase in ["A", "B"]
        },
    }
    if not for_model_setup:
        if DATA_SOURCE == "legacy_random":
            if QUALITY_THRESHOLDS or RUN_HELD_OUT:
                raise ValueError("Legacy fixed-length performance has no task quality or held-out evaluation")
            if LEGACY_EOS not in {"stop", "ignore"}:
                raise ValueError("Unsupported legacy EOS setting")
            identity = {"version": "managed-datasets-v4.1", "source": "legacy_random",
                        "eos": LEGACY_EOS, "thinking": "native completion behavior"}
            if RUN_REAL_GPU:
                dataset_path, source_identity = prepare_legacy(
                    LEGACY_DATASET_PATH, LEGACY_DATASET_REVISION
                )
                settings["base_workload"]["dataset_path"] = str(dataset_path)
                identity.update(source_identity)
        else:
            if DATASETS is None:
                raise ValueError("Run the dataset preparation cell first")
            if RUN_REAL_GPU and DATA_SOURCE == "fixture":
                raise ValueError("Select a real dataset or legacy_random before GPU studies")
            identity = DATASETS.identity
            if dataset_controls() != PREPARED_CONTROLS:
                raise ValueError("Dataset settings changed. Rerun frozen workload preparation")
            settings["base_workload"]["dataset_name"] = "frozen_chat_v4"
            for phase in ("A", "B"):
                doc = DATASETS.validation[phase].document
                if (settings["phase_models"][phase] != doc["model"]
                    or (RUN_REAL_GPU and settings["real_preflight"][phase][FRAMEWORKS[0]]
                        ["model_revision"] != doc["model_revision"])):
                    raise ValueError("Model settings changed after dataset preparation")
        for preflight in settings["real_preflight"].values():
            preflight[FRAMEWORKS[0]]["dataset_v4"] = identity
    runner = TRTLLMRunnerV3(
        settings, BASE_SPACES, EXPANDED_CHOICES, EXTRA_SPACES, RUN_ROOT,
        precision_choices=PRECISION_CHOICES,
        optional_params=optional_params, telemetry_options=TELEMETRY,
        requests_by_concurrency=REQUESTS_BY_CONCURRENCY,
        pruning_min_completed_requests=PRUNING_MIN_COMPLETED_REQUESTS,
    )
    return attach_campaign(runner, DATASETS) if not for_model_setup and DATASETS is not None and DATA_SOURCE != 'legacy_random' else runner

# Construct from the current controls for each invocation; SQLite resumes studies.
def get_runner():
    return make_runner()

def run_objective_group(phase, target, concurrencies, pruner_name="nop", sampler_names=None, constrained=None):
    global LAST_RUN_DIR
    runner = get_runner()
    LAST_RUN_DIR = runner.run_dir
    print("Campaign:", CAMPAIGN, "| W&B run prefix:", RUN_NAME)
    print("Results directory:", LAST_RUN_DIR)
    print("Concurrency / full-trial requests:",
          [(c, REQUESTS_BY_CONCURRENCY[c]) for c in concurrencies])
    return runner.run_objective_group(
        phase, target, concurrencies, pruner_name, sampler_names=sampler_names,
        constrained=CONSTRAINED if constrained is None else constrained,
    )

def setup_pinned_models():
    """Explicit GPU setup; commit resolved profiles into the editable controls."""
    global PHASE_B_MODEL, PHASE_B_SNAPSHOT, MODEL_SETUP_IDENTITY, MODEL_SETUP_CHOICES
    if not RUN_REAL_GPU:
        raise ValueError("Set RUN_REAL_GPU=True before model setup")
    runner = make_runner(for_model_setup=True)
    runner.lab.RUN_REAL_GPU = True
    with runner.lock():
        summary = runner.lab.configure_hf_models(**HF_SETUP)
    framework = HF_SETUP["framework"]
    import copy
    profile = runner.lab.REAL_PREFLIGHT["B", framework]
    # Verification declarations remain editable in PREFLIGHT after setup.
    MODEL_PROFILES[framework] = {
        key: copy.deepcopy(profile[key]) for key in (
            "model_revision", "hf_target_model_id", "fixed_server", "draft_model_id",
            "draft_revision", "speculative_decoding", "framework_version", "cuda_version",
            "driver_version", "gpu_type",
        ) if key in profile
    }
    if profile["expanded_choices"] != EXPANDED_CHOICES[framework]:
        MODEL_PROFILES[framework]["expanded_choices"] = copy.deepcopy(profile["expanded_choices"])
    MODEL_SETUP_CHOICES = copy.deepcopy(EXPANDED_CHOICES[framework])
    PHASE_B_MODEL, PHASE_B_SNAPSHOT = summary["target_model_id"], summary["target_path"]
    MODEL_SETUP_IDENTITY = dict(HF_SETUP)
    return summary


In [ ]:
# Attach the adapter to every notebook-created runner, including model setup.
# Preserve the original factory so this cell is safe to rerun.
_original_make_runner = getattr(make_runner, "_without_trt140_adapter", make_runner)

def make_runner(for_model_setup=False):
    if FRAMEWORKS != ["trtllm"]:
        raise ValueError("This notebook targets a dedicated TRT-LLM VM")
    PREFLIGHT["trtllm"]["notebook_compatibility"] = dict(TRT140_COMPATIBILITY)
    if RUN_REAL_GPU:
        validate_trt140_environment()
    runner = _original_make_runner(for_model_setup=for_model_setup)
    runner.lab.precision_config = MethodType(notebook_precision_config, runner.lab)
    runner.lab.serialize_engine_config = MethodType(notebook_serialize_engine_config, runner.lab)
    original_logger = runner.lab.NotebookWandbLogger
    run_prefix = RUN_NAME

    def named_logger(study, contract):
        logger = original_logger(study, contract)
        if logger.tracker is not None and run_prefix:
            logger.tracker.run.name = f"{run_prefix}-{study.study_name}"
        return logger

    runner.lab.NotebookWandbLogger = named_logger
    return runner

make_runner._without_trt140_adapter = _original_make_runner


In [ ]:
# CPU mapping checks. Checkpoint identity implements Phase B weight precision.
from types import SimpleNamespace
_probe = SimpleNamespace()
for _weights in ("bf16", "fp8"):
    for _cache in ("bf16", "fp8"):
        assert notebook_precision_config(_probe, "trtllm", _weights, _cache) == {"dtype": "bfloat16"}
print("Mapping checks passed; checkpoint preparation validates actual FP8 storage.")


## Resolve a pinned Hugging Face model for TRT-LLM
Run these cells after the configuration and runner cells. Setup downloads or reuses the model snapshot and requires a CUDA GPU. Choose an immutable commit for reproducible reruns; `main` resolves to its current commit. The setup below selects ordinary serving.


In [ ]:
if RUN_REAL_GPU and RUN_MODEL_SETUP:
    import tempfile
    from pathlib import Path

    hf_cache = Path(tempfile.mkdtemp(prefix="hf-cache-", dir="/tmp"))
    HF_SETUP["cache_dir"] = str(hf_cache)

    print("Hugging Face cache:", hf_cache)
    print("Writable:", hf_cache.is_dir())


In [ ]:
if RUN_REAL_GPU and RUN_MODEL_SETUP:
    hf_models = setup_pinned_models()
    hf_models



In [ ]:
if RUN_REAL_GPU and RUN_MODEL_SETUP:
    MODEL = hf_models["target_model_id"]
    MODEL_SNAPSHOT = hf_models["target_path"]
    for key in ("gpu_type", "framework_version", "cuda_version",
                "driver_version", "model_revision"):
        PREFLIGHT["trtllm"][key] = MODEL_PROFILES["trtllm"][key]
    print("Pinned snapshot:", MODEL_SNAPSHOT)
    print("Pinned revision:", PREFLIGHT["trtllm"]["model_revision"])



In [ ]:
if RUN_REAL_GPU and RUN_MODEL_SETUP:
    MODEL = hf_models["target_model_id"]
    MODEL_SNAPSHOT = hf_models["target_path"]

    # Record the resolved runtime and revision for Phase A.
    for key in (
        "gpu_type",
        "framework_version",
        "cuda_version",
        "driver_version",
        "model_revision",
    ):
        PREFLIGHT["trtllm"][key] = MODEL_PROFILES["trtllm"][key]

    print("Phase A snapshot:", MODEL_SNAPSHOT)
    print("Phase B snapshot:", PHASE_B_SNAPSHOT)


## Prepare or reuse FP8 model weights
Set `MODELOPT_ROOT` to an installed ModelOpt checkout and `RUN_FP8_EXPORT=True` once.
The subprocess calibrates the pinned source and exports real FP8 tensors. Keep
`RUN_FP8_EXPORT=False` on subsequent runs. Export completion is not quality approval.


In [ ]:
if RUN_REAL_GPU and True:
    # Run after resolving the pinned BF16 source, before CLI audit or studies.
    # Install ModelOpt's hf example requirements in the TRT environment first.
    # Both current examples/hf_ptq and older examples/llm_ptq layouts are supported.
    import re
    import shlex
    import subprocess

    if PRECISION_CHOICES["trtllm"]["weight_precision"] == ["fp8"]:
        export_dir = Path(TRT_FP8_CHECKPOINT).resolve()
        source = Path(MODEL_SNAPSHOT).resolve()
        if RUN_FP8_EXPORT:
            if export_dir.exists() and any(export_dir.iterdir()):
                raise ValueError("Export directory is not empty; reuse it or choose a new path")
            scripts = [Path(MODELOPT_ROOT) / "examples" / layout / "hf_ptq.py"
                       for layout in ("hf_ptq", "llm_ptq")]
            script = next((s for s in scripts if s.is_file()), None)
            if script is None:
                raise ValueError("Set MODELOPT_ROOT to an existing NVIDIA Model-Optimizer checkout")
            # Record the exact checkout; never install/upgrade dependencies in benchmark cells.
            revision = subprocess.check_output(["git", "-C", str(MODELOPT_ROOT), "rev-parse", "HEAD"], text=True).strip()
            help_result = subprocess.run(
                [sys.executable, str(script), "--help"], cwd=script.parent,
                text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            if help_result.returncode:
                print(help_result.stdout, flush=True)
                raise RuntimeError(
                    "ModelOpt export script failed before calibration. See output above; "
                    "check example dependencies against the installed ModelOpt/TRT environment "
                    "before changing packages.")
            help_text = help_result.stdout
            command = [sys.executable, str(script), "--pyt_ckpt_path", str(source),
                       "--qformat", "fp8", "--kv_cache_qformat", "fp8",
                       "--export_path", str(export_dir), "--calib_size", str(FP8_CALIB_SAMPLES),
                       "--batch_size", "1", "--dataset", FP8_CALIB_DATASET]
            if "--calib_seq" in help_text:
                command += ["--calib_seq", str(FP8_CALIB_SEQUENCE_LENGTH)]
            else:
                raise ValueError("ModelOpt example lacks --calib_seq; use a compatible checkout")
            if "--export_fmt" in help_text:
                command += ["--export_fmt", "hf"]
            advertised = set(re.findall(r"--[a-z][a-z0-9_-]*", help_text))
            missing = {x for x in command if x.startswith("--")} - advertised
            if missing:
                raise ValueError(f"ModelOpt example lacks required options: {sorted(missing)}")
            print("One-time calibration/export:", shlex.join(command))
            # Same VM lock as optimization; prevent concurrent GPU owners.
            preparation_runner = make_runner(for_model_setup=True)
            with preparation_runner.lock():
                subprocess.run(command, cwd=script.parent, check=True)
            identity = _fp8_export_metadata(export_dir)
            manifest = {"source_model": MODEL, "source_snapshot": str(source),
                        "source_revision": PREFLIGHT["trtllm"]["model_revision"],
                        "source_config_sha256": _file_sha(source / "config.json"),
                        "modelopt_version": version("nvidia-modelopt"),
                        "modelopt_commit": revision, "export_command": command,
                        "checkpoint": identity}
            safe_write_json(export_dir / "phase1-fp8-export.json", manifest)
        artifact = validate_fp8_checkpoint(export_dir, source)
        print("FP8 checkpoint:", export_dir)
        print("Artifact:", artifact["checkpoint"]["artifact_id"])
        print("Now run the CLI audit and enable the GPU smoke test before approving fp8/fp8.")
    else:
        print("BF16 weight campaign selected; no FP8 export required.")



## Installed TRT-LLM CLI audit and optional GPU smoke test
Run after pinned-model setup. Check native flag names, nested schema values, and
both KV precision settings before approving preflight. The optional smoke cell starts
one server, prints a response, and cleans up. Repeat with BF16 and FP8 cache.
The audit does not establish model/kernel support or output quality.


In [ ]:
if RUN_REAL_GPU and RUN_NATIVE_AUDIT:
    import re
    import shlex
    import shutil
    import subprocess
    validate_trt140_environment()
    TRT_EXECUTABLE = resolve_notebook_trt_cli()
    help_text = runtime_help()  # executes trtllm-serve serve --help
    TRT_SERVE_OPTIONS = set(re.findall(r"--[a-z][a-z0-9_-]*", help_text))
    print("CLI:", TRT_EXECUTABLE, "version:", version("tensorrt_llm"))

    def notebook_trt_command(lab, config, port=8000):
        model = TRT_FP8_CHECKPOINT if PRECISION_CHOICES["trtllm"]["weight_precision"] == ["fp8"] else MODEL_SNAPSHOT
        command = native_command(lab, config, model, 8, port, executable=TRT_EXECUTABLE)
        audit_command(command, help_text)
        return command

    audit_runner = make_runner(for_model_setup=True)
    audit_runner.lab.RUN_REAL_GPU = True  # validate installed schemas, without launching a server
    for cache in PRECISION_CHOICES["trtllm"]["kv_cache_precision"]:
        for policy in EXPANDED_CHOICES["trtllm"]["capacity_scheduler_policy"]:
            for graph in (False, True):
                config = audit_runner.lab.render_engine_config("trtllm", 8, {
                    "max_batch_size": 8, "max_num_tokens": 2048,
                    "kv_cache_free_gpu_memory_fraction": 0.75,
                })
                config.update(audit_runner.lab.precision_config("trtllm", "bf16", cache))
                config["kv_cache_config"].update(dtype="fp8" if cache == "fp8" else "auto",
                                                 enable_block_reuse=False)
                config.update(attn_backend="TRTLLM", enable_chunked_prefill=True,
                              scheduler_config={"capacity_scheduler_policy": policy},
                              cuda_graph_config={"batch_sizes": [1, 2, 4, 8], "enable_padding": True} if graph else None)
                print(cache, policy, graph, notebook_trt_command(audit_runner.lab, config))
    print("CLI and nested-schema audit passed. GPU/model support still requires a smoke test.")

    # Exercise enabled optional controls through the actual sampler and serializer.
    import optuna
    for concurrency in CONCURRENCIES:
        for seed in (11, 29, 47):
            study = optuna.create_study(sampler=optuna.samplers.RandomSampler(seed=seed))
            trial = study.ask()
            config = audit_runner.lab.suggest_config(trial, "trtllm", concurrency, "expanded")
            native = audit_runner.lab.serialize_engine_config("trtllm", concurrency, config)
            command = native_command(audit_runner.lab, config, MODEL_SNAPSHOT, concurrency,
                                     executable=TRT_EXECUTABLE)
            audit_command(command, help_text)
    print("Enabled optional schema/CLI audit passed; GPU execution is not validated.")



In [ ]:
if RUN_REAL_GPU and True:
    # Optional: starts ONE real server, requests text, then stops it; no Optuna trial.
    RUN_TRT140_SMOKE = False
    SMOKE_WEIGHTS, SMOKE_CACHE = PRECISION_CHOICES["trtllm"]["weight_precision"][0], "fp8"  # repeat with bf16/bf16 as a reference
    SMOKE_PORT = 8000  # must be unused
    if RUN_TRT140_SMOKE:
        if [SMOKE_WEIGHTS] != PRECISION_CHOICES["trtllm"]["weight_precision"]:
            raise ValueError("SMOKE_WEIGHTS must match this campaign; change the weight control to compare")
        if SMOKE_WEIGHTS == "fp8":
            validate_fp8_checkpoint(TRT_FP8_CHECKPOINT, MODEL_SNAPSHOT)
        import requests
        from llm_optimizer.server_utils import start_server, terminate_process_top_down

        smoke_runner = make_runner(for_model_setup=True)
        smoke_lab = smoke_runner.lab
        smoke_config = smoke_lab.render_engine_config("trtllm", 8, {
            "max_batch_size": 8, "max_num_tokens": 2048,
            "kv_cache_free_gpu_memory_fraction": 0.75,
        })
        smoke_config.update(smoke_lab.precision_config("trtllm", SMOKE_WEIGHTS, SMOKE_CACHE))
        smoke_config["kv_cache_config"].update(
            dtype="fp8" if SMOKE_CACHE == "fp8" else "auto", enable_block_reuse=False)
        smoke_config.update(attn_backend="TRTLLM", enable_chunked_prefill=True,
                            cuda_graph_config=None)
        smoke_config = {**MODEL_PROFILES["trtllm"].get("fixed_server", {}), **smoke_config}
        smoke_command = notebook_trt_command(smoke_lab, smoke_config, SMOKE_PORT)
        print(smoke_command)
        with smoke_runner.lock():
            process = None
            try:
                process = start_server(smoke_command, {},
                                       f"http://127.0.0.1:{SMOKE_PORT}/health", False)
                response = requests.post(f"http://127.0.0.1:{SMOKE_PORT}/v1/chat/completions",
                    json={"model": str(TRT_FP8_CHECKPOINT if SMOKE_WEIGHTS == "fp8" else MODEL_SNAPSHOT), "messages": [
                        {"role": "user", "content": "What is 17 times 19? Explain briefly."}],
                        "temperature": 0, "max_tokens": 256}, timeout=180)
                response.raise_for_status()
                print(response.json()["choices"][0]["message"])
            finally:
                if process is not None:
                    terminate_process_top_down(process)
        print("Smoke completed. Review text; this does not approve the expanded search space.")



Set the native configuration and quality verification fields in `PREFLIGHT["trtllm"]`
only after checking this VM. List both selected BF16-weight/KV-cache combinations,
or narrow the cache choices to the combination you verified. Rerunning initial
controls resets model setup. Notebook adapters do not apply to the batch script.


In [ ]:
if RUN_REAL_GPU and RUN_NATIVE_AUDIT:
    print("TRT-LLM version:", MODEL_PROFILES["trtllm"]["framework_version"])
    print("Expanded choices:", EXPANDED_CHOICES["trtllm"])
    PRECISION_CHOICES["trtllm"] = {
        "weight_precision": ["fp8"],
        "kv_cache_precision": ["fp8"],
    }
    # Set True and list pairs only after the corresponding VM checks pass.
    PREFLIGHT["trtllm"].update({
        "native_config_verified": False,
        "quality_verified": False,
        "verified_precision_combinations": [
            # "bf16/bf16",
            # "bf16/fp8",
            # "fp8/bf16",
            # "fp8/fp8",
        ],
    })
    print("Preflight:", PREFLIGHT["trtllm"])
    print("Precision choices:", PRECISION_CHOICES["trtllm"])

    print("Enabled optional parameters:", {k: v for k, v in optional_params.items() if v["enabled"]})



## Frozen workload preparation and CPU preflight

In [ ]:
# Prepare and verify the complete dataset campaign before optimization.
def dataset_controls():
    return copy.deepcopy((
        DATA_SOURCE, SOURCE_FORMAT, DATASET_ID, DATASET_REVISION, DATASET_SPLIT,
        DATASET_CONFIG, GENERATION, SCORER, SCORE_QUALITY, QUALITY_THRESHOLDS,
        SAMPLING_SEED, TOKEN_BUCKETS_BY_CONCURRENCY, HELD_OUT_COUNT,
        DATASET_REPEAT_REQUESTS, VALIDATION_SEED, VALIDATION_REPEATS,
        THINKING_TEMPLATE_VERIFIED, FRAMEWORKS, REQUESTS_BY_CONCURRENCY,
        MODEL, MODEL_REVISION, MODEL_SNAPSHOT, PHASE_B_MODEL, PHASE_B_SNAPSHOT,
    ))

PREPARED_CONTROLS = dataset_controls()
if DATA_SOURCE != "legacy_random":
    models = {"A": MODEL, "B": PHASE_B_MODEL or MODEL}
    snapshots = {"A": MODEL_SNAPSHOT, "B": PHASE_B_SNAPSHOT or MODEL_SNAPSHOT}
    if RUN_REAL_GPU:
        models["B"], snapshots["B"] = phase_b_artifact()
    revisions = {p: MODEL_REVISION for p in models}
    if DATA_SOURCE == "fixture":
        revisions = dict.fromkeys(models, "0" * 40)  # offline fixture sentinel
    elif RUN_REAL_GPU:
        revisions = {
            "A": PREFLIGHT[FRAMEWORKS[0]]["model_revision"],
            "B": MODEL_PROFILES.get(FRAMEWORKS[0], {}).get(
                "model_revision", PREFLIGHT[FRAMEWORKS[0]]["model_revision"]
            ),
        }
    DATASETS = CampaignWorkloads(
        source=DATA_SOURCE, source_format=SOURCE_FORMAT, dataset_id=DATASET_ID,
        revision=DATASET_REVISION, split=DATASET_SPLIT, dataset_config=DATASET_CONFIG,
        generation=GENERATION, scorer=SCORER, thresholds=QUALITY_THRESHOLDS,
        framework=FRAMEWORKS[0], models=models, revisions=revisions, snapshots=snapshots,
        requests=REQUESTS_BY_CONCURRENCY, repeat_count=DATASET_REPEAT_REQUESTS,
        repeat_seeds=range(VALIDATION_SEED, VALIDATION_SEED + VALIDATION_REPEATS),
        held_out_count=HELD_OUT_COUNT, sampling_seed=SAMPLING_SEED,
        buckets=TOKEN_BUCKETS_BY_CONCURRENCY, thinking_verified=THINKING_TEMPLATE_VERIFIED,
        fixture_count=max(max(REQUESTS_BY_CONCURRENCY.values()), DATASET_REPEAT_REQUESTS)
                      + HELD_OUT_COUNT,
        score_quality=SCORE_QUALITY,
    )
else:
    DATASETS = None
if DATASETS is not None:
    print("Frozen tuning manifests:", DATASETS.identity["training"])
    print("Frozen held-out manifests:", DATASETS.identity["held_out"])
cpu_preflight_runner = make_runner(for_model_setup=True)
for phase in ("A", "B"):
    for concurrency in CONCURRENCIES:
        assert cpu_preflight_runner.lab.sampling_menu(
            FRAMEWORKS[0], concurrency, "baseline" if phase == "A" else "expanded"
        )

## Independent experiment cells
Day 1: preflight, untuned baseline measurements, and recovery drills. Days 2–4: base (A) versus extended (B) searches on the same model, then median versus no pruning. Days 5–6: independent repeats and workload checks. Day 7: sustained-load confirmation. The queue below performs the search stage; use the validation cell afterwards. Measure trial duration during the pilot and adjust trial counts to fit the week. A finite completed queue exits successfully rather than repeating identical experiments indefinitely.

### A: single e2e (no pruning)

In [ ]:
if RUN_REAL_GPU and RUN_STUDIES:
    A_single_e2e_studies = run_objective_group(
        "A", "e2e", [1], pruner_name="nop",
    )
    plot_group(A_single_e2e_studies)



### A: single ttft (no pruning)

In [ ]:
if RUN_REAL_GPU and RUN_STUDIES:
    A_single_ttft_studies = run_objective_group(
        "A", "ttft", [1], pruner_name="nop",
    )
    plot_group(A_single_ttft_studies)



### A: loaded throughput (no pruning)

In [ ]:
if RUN_REAL_GPU and RUN_STUDIES:
    A_loaded_throughput_studies = run_objective_group(
        "A", "throughput", [c for c in CONCURRENCIES if c > 1], pruner_name="nop",
    )
    plot_group(A_loaded_throughput_studies)



### A: loaded e2e (no pruning)

In [ ]:
if RUN_REAL_GPU and RUN_STUDIES:
    A_loaded_e2e_studies = run_objective_group(
        "A", "e2e", [c for c in CONCURRENCIES if c > 1], pruner_name="nop",
    )
    plot_group(A_loaded_e2e_studies)



### B: single e2e (no pruning)

In [ ]:
if RUN_REAL_GPU and RUN_STUDIES:
    B_single_e2e_studies = run_objective_group(
        "B", "e2e", [1], pruner_name="nop",
    )
    plot_group(B_single_e2e_studies)



### B: single ttft (no pruning)

In [ ]:
if RUN_REAL_GPU and RUN_STUDIES:
    B_single_ttft_studies = run_objective_group(
        "B", "ttft", [1], pruner_name="nop",
    )
    plot_group(B_single_ttft_studies)



### B: loaded throughput (no pruning)

In [ ]:
if RUN_REAL_GPU and RUN_STUDIES:
    B_loaded_throughput_studies = run_objective_group(
        "B", "throughput", [c for c in CONCURRENCIES if c > 1], pruner_name="nop",
    )
    plot_group(B_loaded_throughput_studies)



### B: loaded e2e (no pruning)

In [ ]:
if RUN_REAL_GPU and RUN_STUDIES:
    B_loaded_e2e_studies = run_objective_group(
        "B", "e2e", [c for c in CONCURRENCIES if c > 1], pruner_name="nop",
    )
    plot_group(B_loaded_e2e_studies)



### B: loaded throughput with median pruning
The first ten trials establish pruning history. Decisions use measured request checkpoints, with at least three reference trials. The no-pruning comparison above uses the same workload and seeds.

In [ ]:
if RUN_REAL_GPU and RUN_STUDIES:
    B_loaded_throughput_studies = run_objective_group(
        "B", "throughput", [c for c in CONCURRENCIES if c > 1],
        pruner_name="median",
    )
    plot_group(B_loaded_throughput_studies)



## View and export saved model responses
Run this cell after any experiment has saved a benchmark result. By default it shows every
response from the most recently modified result file's last benchmark. Select a file and row
to inspect an earlier trial. It also writes a readable text copy alongside the JSONL file.


In [ ]:
# Run after a benchmark finishes; rerun to inspect newer saved results.
# Optional: paste a prior campaign directory or an exact *-measured.jsonl path.
RESULTS_DIR = None
RESULT_FILE = None
RESULT_INDEX = -1  # last saved benchmark in the selected file
WRITE_READABLE_COPY = True

if RESULT_FILE is None and RESULTS_DIR is None and "LAST_RUN_DIR" not in globals():
    print("No saved benchmark results yet.")
    output_path = None
elif RESULT_FILE is not None:
    output_path = Path(RESULT_FILE)
else:
    output_dir = Path(RESULTS_DIR) if RESULTS_DIR is not None else globals().get("LAST_RUN_DIR")
    if output_dir is None:
        raise ValueError("Run an experiment first, or set RESULTS_DIR / RESULT_FILE above")
    candidates = list(Path(output_dir).glob("*-measured.jsonl"))
    output_path = max(candidates, key=lambda p: p.stat().st_mtime) if candidates else None

if output_path is None:
    print("No saved benchmark results yet.")
else:
    rows = []
    for line in output_path.read_text().splitlines():
        if line.strip():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                print("Skipping an incomplete/invalid JSONL row; rerun after the writer finishes.")
    if not rows:
        print("No complete result rows yet.")
    else:
        row = rows[RESULT_INDEX]
        details = row.get("results", {})
        texts = details.get("generated_texts", [])
        print("File:", output_path, "| selected row:", RESULT_INDEX, "| rows:", len(rows))
        print("Trial:", row.get("optimization", {}).get("trial_number"))
        if not texts:
            print("This result contains no saved text; enable SAVE_REQUEST_OUTPUTS for future runs.")
        else:
            errors = details.get("errors", [])
            blocks = []
            for i, text in enumerate(texts):
                block = f"--- Request {i + 1} ---\n{text}"
                if i < len(errors) and errors[i]:
                    block += f"\nError: {errors[i]}"
                blocks.append(block)
            readable = "\n\n".join(blocks)
            print(readable)
            if WRITE_READABLE_COPY:
                row_number = RESULT_INDEX if RESULT_INDEX >= 0 else len(rows) + RESULT_INDEX
                readable_path = output_path.with_name(
                    f"{output_path.stem}-row{row_number}-generations.txt")
                readable_path.write_text(readable + "\n")
                print("Readable copy saved:", readable_path)


## Independent candidate validation
Run after search, within the remaining GPU budget. This uses the repository’s shortlist and independent validation workload/repeats; it is not an automatic sustained-load soak test. Keep the serving stack pinned and record any separate soak workload. Validation in this interactive cell does not use the batch watchdog.

In [ ]:
if RUN_REAL_GPU and RUN_VALIDATION:
    from llm_optimizer.experiment.workflow import load_studies, write_json
    runner = get_runner()
    with runner.lock():
        studies = load_studies(runner.lab)
        validation = runner.lab.validate_candidates(runner.lab.shortlist(studies))
        write_json(runner.run_dir / "validation.json", runner.lab.artifact_value(validation))


## Execution boundary

The v3 batch CLI reads its own legacy settings. For this campaign, run these v4 notebook cells interactively or with a notebook executor after setting the controls and completing runtime verification. The original GPU lock, SQLite resume and arm exports apply. After abrupt kernel termination, verify that its servers stopped before resuming. V3 candidate repeats and held-out task evaluation are separately enabled.

## Optional Pareto, constraint, and pruning experiments
Run these cells independently after editing SLOs, samplers and pinned serving settings. Active pruning is supported only for unconstrained scalar objectives. Calibration uses the same task workload and scaled request checkpoints; pilot samples are additional to the 20-attempt comparison budgets. Diagnostics replay a first-checkpoint median rule and estimate savings; validate them on the GPU before trusting pruning. Threshold searches reuse a persisted task calibration.


In [ ]:
if RUN_REAL_GPU and RUN_STUDIES:
    pareto_latency = run_objective_group("B", "pareto_latency", [1])
    plot_group(pareto_latency)



In [ ]:
if RUN_REAL_GPU and RUN_STUDIES:
    pareto_loaded = run_objective_group("B", "pareto_throughput", [c for c in CONCURRENCIES if c > 1])
    plot_group(pareto_loaded)



In [ ]:
if RUN_REAL_GPU and RUN_STUDIES:
    constrained_scalar = run_objective_group("B", "e2e", [1], constrained=True)
    plot_group(constrained_scalar)



In [ ]:
if RUN_REAL_GPU and RUN_STUDIES:
    constrained_pareto = run_objective_group("B", "pareto_latency", [1], constrained=True)
    plot_group(constrained_pareto)



In [ ]:
if RUN_REAL_GPU and RUN_STUDIES:
    diagnostics = get_runner().calibrate("B", "throughput", 8, FRAMEWORKS[0], samples=PRUNING_SAMPLES)
    diagnostics



In [ ]:
if RUN_REAL_GPU and RUN_STUDIES:
    pruning_comparison = {name: run_objective_group("B", "throughput", [8], pruner_name=name)
                          for name in PRUNER_NAMES}



## Held-out evaluation of a selected study winner

Set `HELD_OUT_STUDY_NAME` to a completed arm from this campaign. Scalar selection honors the study direction and its SLO constraints. For a Pareto study, nominate an eligible front member in `HELD_OUT_PARETO_TRIAL`. The winner runs once at its study concurrency on the separate frozen manifest.

Inference and live pruning use v3 performance measurements. Quality is computed after cleanup from the measured samples. When thresholds are set, a failed final quality gate leaves the trial pruned and ineligible for selection. It can still have reported performance checkpoints. Held-out results stay separate from both tuning and v3 repeat validation.

In [ ]:
if RUN_REAL_GPU and RUN_HELD_OUT:
    import optuna

    runner = get_runner()
    study = optuna.load_study(study_name=HELD_OUT_STUDY_NAME, storage=runner.lab.STORAGE)
    held_out_result = evaluate_held_out(runner, study, pareto_trial=HELD_OUT_PARETO_TRIAL)
    if held_out_result["measurement"] is None:
        print("Held-out benchmark failed:", held_out_result["failure"])
    else:
        print(held_out_result["measurement"]["execution"]["dataset_v4"])